In [1]:
import torch
torch.cuda.empty_cache()


In [2]:
import imageio.v3 as iio
import cv2
from matplotlib import pyplot as plt
from IPython.display import Image, Video
from openface.face_detection import FaceDetector

# The next block fixes a bug that causes crashes on Unix-based systems.
# You should apply this patch before importing "LandmarkDetector" from openface.

# --- begin of patch ---
# Patching LandmarkDetector hardcoded log folder
from openface.STAR.demo import Alignment

# Access the utility module through Alignment.__init__'s globals
utility = Alignment.__init__.__globals__['utility']

# Only patch once to avoid recursion
if getattr(utility.set_environment, '__name__', '') != 'patched_set_environment':
    original_set_environment = utility.set_environment

    def patched_set_environment(config):
        result = original_set_environment(config)
        config.log_dir = '.'
        return result

    utility.set_environment = patched_set_environment
# --- end of patch ---

from openface.landmark_detection import LandmarkDetector
from openface.multitask_model import MultitaskPredictor
import torch.nn.functional as F
import torch
import numpy as np
from typing import Union
import os
import imageio.v3 as iio
import cv2
from matplotlib import pyplot as plt
from IPython.display import Image, Video
from openface.face_detection import FaceDetector

# The next block fixes a bug that causes crashes on Unix-based systems.
# You should apply this patch before importing "LandmarkDetector" from openface.

# --- begin of patch ---
# Patching LandmarkDetector hardcoded log folder
from openface.STAR.demo import Alignment

# Access the utility module through Alignment.__init__'s globals
utility = Alignment.__init__.__globals__['utility']

# Only patch once to avoid recursion
if getattr(utility.set_environment, '__name__', '') != 'patched_set_environment':
    original_set_environment = utility.set_environment

    def patched_set_environment(config):
        result = original_set_environment(config)
        config.log_dir = '.'
        return result

    utility.set_environment = patched_set_environment
# --- end of patch ---

from openface.landmark_detection import LandmarkDetector
from openface.multitask_model import MultitaskPredictor
import torch.nn.functional as F
import torch
import numpy as np
from typing import Union
import os


# The next patch overrides a function in FaceDetector which expects a filename as input to also accept numpy arrays
# This will be useful later when we want to pass webcam frames directly without saving to disk

# --- begin of patch ---
def override_preprocess_image(self, image_path: Union[str, np.ndarray], resize: float = 1.0):
    if isinstance(image_path, (str, os.PathLike)):
        img_raw = cv2.imread(str(image_path), cv2.IMREAD_COLOR)  # BGR, 3 channels
        if img_raw is None:
            raise ValueError(f"Failed to read image from path: {image_path}")
    elif isinstance(image_path, np.ndarray):
        img_raw = image_path
    else:
        raise TypeError("image_path must be a str/Path-like path or a numpy.ndarray (BGR frame).")

    if img_raw.ndim == 2:
        # Grayscale -> BGR
        img_raw = cv2.cvtColor(img_raw, cv2.COLOR_GRAY2BGR)
    elif img_raw.ndim == 3:
        if img_raw.shape[2] == 4:
            # BGRA -> BGR
            img_raw = cv2.cvtColor(img_raw, cv2.COLOR_BGRA2BGR)
        elif img_raw.shape[2] != 3:
            raise ValueError(f"Unsupported channel count: {img_raw.shape[2]} (expected 1, 3, or 4)")
    else:
        raise ValueError(f"Unsupported image shape {img_raw.shape}; expected HxW or HxWxC.")

    # --- Preprocess as in original code
    img = img_raw.astype(np.float32, copy=False)
    if resize != 1.0:
        img = cv2.resize(img, None, fx=resize, fy=resize, interpolation=cv2.INTER_LINEAR)

    # Mean subtraction in BGR (matching many Caffe-style models)
    img -= (104.0, 117.0, 123.0)

    # Ensure contiguous before transpose (safer with slices or unusual strides)
    img = np.ascontiguousarray(img.transpose(2, 0, 1))  # (C, H, W)

    img = torch.from_numpy(img).unsqueeze(0).to(self.device)  # (1, C, H, W)

    return img, img_raw

FaceDetector.preprocess_image = override_preprocess_image
# --- end of patch ---
import os
import time
import csv
import uuid
from datetime import datetime

import cv2
import torch.nn.functional as F
import numpy as np


device = "cpu"  

face_model_path = "./weights/Alignment_RetinaFace.pth"
multitask_model_path = "./weights/MTL_backbone.pth"


emo_names = ["Neutral", "Happy", "Sad", "Surprise",
             "Fear", "Disgust", "Anger", "Contempt"]


det = FaceDetector(face_model_path, device=device)
mtl = MultitaskPredictor(model_path=multitask_model_path, device=device)

SAVE_DIR = "./data/captures"
os.makedirs(SAVE_DIR, exist_ok=True)

LABEL_CSV = "./data/labels.csv"
os.makedirs(os.path.dirname(LABEL_CSV), exist_ok=True)

if not os.path.exists(LABEL_CSV):
    with open(LABEL_CSV, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "timestamp", "filename", "emotion_idx", "emotion_name", "prob",
            "x1", "y1", "x2", "y2"
        ])

def process_frame(frame):
    cropped_face, dets = det.get_face(frame)
    vis = frame.copy()

    if dets is None or len(dets) == 0:
        return vis, None, None

    dets = np.array(dets)
    scores = dets[:, 4] if dets.shape[1] > 4 else np.ones(len(dets))
    best_idx = int(np.argmax(scores))
    best_det = dets[best_idx]

    x1, y1, x2, y2 = map(int, best_det[:4])
    h, w = frame.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w - 1, x2), min(h - 1, y2)

    face_crop = frame[y1:y2, x1:x2]
    if face_crop.size == 0:
        return vis, None, None

    emotion_logits, _, _ = mtl.predict(face_crop)
    probs = F.softmax(emotion_logits, dim=1)[0].cpu().numpy()
    emo_idx = int(np.argmax(probs))
    emo_prob = float(probs[emo_idx])
    emo_name = emo_names[emo_idx]
    emo_label = f"{emo_name} ({emo_prob*100:.1f}%)"

    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(vis, emo_label, (x1, max(0, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3, cv2.LINE_AA)
    cv2.putText(vis, emo_label, (x1, max(0, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)

    info = {
        "emo_idx": emo_idx,
        "emo_name": emo_name,
        "emo_prob": emo_prob,
        "bbox": (x1, y1, x2, y2),
    }

    return vis, face_crop, info

cam = cv2.VideoCapture(0)
if not cam.isOpened():
    raise SystemExit("Failed to open camera")

cv2.namedWindow("Webcam Emotion (ESC to exit)", cv2.WINDOW_NORMAL)

last_save_time = 0.0
SAVE_INTERVAL = 5.0

while True:
    ret, frame = cam.read()
    if not ret:
        print("Failed to read frame from camera")
        break

    vis, face_crop, info = process_frame(frame)

    if face_crop is not None and info is not None:
        now = time.time()
        if now - last_save_time >= SAVE_INTERVAL:
            last_save_time = now

            ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"{ts_str}_{info['emo_name']}_{uuid.uuid4().hex[:8]}.jpg"
            save_path = os.path.join(SAVE_DIR, filename)

            cv2.imwrite(save_path, face_crop)

            x1, y1, x2, y2 = info["bbox"]
            with open(LABEL_CSV, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([
                    ts_str,
                    filename,
                    info["emo_idx"],
                    info["emo_name"],
                    f"{info['emo_prob']:.4f}",
                    x1, y1, x2, y2
                ])

            print(f"[Saved] {filename} -> {info['emo_name']} ({info['emo_prob']*100:.1f}%)")

    cv2.imshow("Webcam Emotion (ESC to exit)", vis)

    key = cv2.waitKey(1) & 0xFF
    if key == 27:
        break

cam.release()
cv2.destroyAllWindows()


c:\Users\Lenovo\robot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading pretrained model from ./weights/Alignment_RetinaFace.pth
remove prefix 'module.'
Missing keys:0
Unused checkpoint keys:0
Used keys:300


c:\Users\Lenovo\robot\Lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b0_ns to current tf_efficientnet_b0.ns_jft_in1k.
  model = create_fn(


Loading multitask model from ./weights/MTL_backbone.pth...
[Saved] 20260322_153537_Neutral_9c1a24c7.jpg -> Neutral (39.4%)
[Saved] 20260322_153542_Happy_6130cd6f.jpg -> Happy (33.8%)
[Saved] 20260322_153547_Neutral_e37bf482.jpg -> Neutral (44.0%)


In [3]:
import os
import csv
import time
import uuid
from datetime import datetime
from typing import Union

import cv2
import joblib
import numpy as np
from PIL import Image

import torch
from transformers import AutoImageProcessor, AutoModel

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

from openface.face_detection import FaceDetector


# =========================================================
# 0) 基本配置
# =========================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DINO_NAME = "facebook/dinov3-vits16plus-pretrain-lvd1689m"


# =========================================================
# 1) Patch: 让 FaceDetector 支持 numpy frame（你原来的思路）
# =========================================================
def override_preprocess_image(self, image_path: Union[str, np.ndarray], resize: float = 1.0):
    if isinstance(image_path, (str, os.PathLike)):
        img_raw = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if img_raw is None:
            raise ValueError(f"Failed to read image from path: {image_path}")
    elif isinstance(image_path, np.ndarray):
        img_raw = image_path
    else:
        raise TypeError("image_path must be a str/Path-like or numpy.ndarray (BGR frame).")

    if img_raw.ndim == 2:
        img_raw = cv2.cvtColor(img_raw, cv2.COLOR_GRAY2BGR)
    elif img_raw.ndim == 3:
        if img_raw.shape[2] == 4:
            img_raw = cv2.cvtColor(img_raw, cv2.COLOR_BGRA2BGR)
        elif img_raw.shape[2] != 3:
            raise ValueError(f"Unsupported channel count: {img_raw.shape[2]}")
    else:
        raise ValueError(f"Unsupported image shape {img_raw.shape}")

    img = img_raw.astype(np.float32, copy=False)
    if resize != 1.0:
        img = cv2.resize(img, None, fx=resize, fy=resize, interpolation=cv2.INTER_LINEAR)

    img -= (104.0, 117.0, 123.0)
    img = np.ascontiguousarray(img.transpose(2, 0, 1))
    img = torch.from_numpy(img).unsqueeze(0).to(self.device)

    return img, img_raw

FaceDetector.preprocess_image = override_preprocess_image


# =========================================================
# 2) DINOv3 Embedder（只用 AutoModel，兼容你当前 transformers）
# =========================================================
class DinoEmbedder:
    def __init__(self, model_name=DINO_NAME, device=DEVICE):
        self.device = device
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device).eval()

    @torch.no_grad()
    def embed_bgr(self, bgr: np.ndarray) -> np.ndarray:
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(rgb)

        inputs = self.processor(images=img, return_tensors="pt").to(self.device)
        out = self.model(**inputs)

        # 常见输出：out.last_hidden_state: [B, N, D]
        if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
            x = out.last_hidden_state  # [1, N, D]
            # 优先 CLS token
            if x.shape[1] >= 1:
                emb = x[:, 0, :]  # [1, D]
            else:
                emb = x.mean(dim=1)
        else:
            # 少数模型用 pooler_output
            if hasattr(out, "pooler_output") and out.pooler_output is not None:
                emb = out.pooler_output
            else:
                raise RuntimeError("Model output has no last_hidden_state/pooler_output, can't build embedding.")

        return emb[0].float().cpu().numpy()


# =========================================================
# 3) 数据采集（手工按键打标签）
# =========================================================
def ensure_csv(label_csv: str):
    os.makedirs(os.path.dirname(label_csv), exist_ok=True)
    if not os.path.exists(label_csv):
        with open(label_csv, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["timestamp", "filename", "emotion_name", "x1", "y1", "x2", "y2"])

def collect_webcam(
    face_weights: str,
    save_dir: str,
    label_csv: str,
    cam_id: int = 0,
    save_interval: float = 2.0,
    default_label: str = "Neutral",
):
    os.makedirs(save_dir, exist_ok=True)
    ensure_csv(label_csv)

    det = FaceDetector(face_weights, device="cpu")
    cap = cv2.VideoCapture(cam_id)
    if not cap.isOpened():
        raise SystemExit("Failed to open camera")

    cv2.namedWindow("Collect (ESC exit)", cv2.WINDOW_NORMAL)

    last_save = 0.0
    current_label = default_label

    help_text = "Keys: 1-8 label | S save now | ESC quit"
    label_map = {
        ord('1'): "Neutral",
        ord('2'): "Happy",
        ord('3'): "Sad",
        ord('4'): "Surprise",
        ord('5'): "Fear",
        ord('6'): "Disgust",
        ord('7'): "Anger",
        ord('8'): "Contempt",
    }

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        vis = frame.copy()
        cropped_face, dets = det.get_face(frame)

        face_crop = None
        bbox = None
        if dets is not None and len(dets) > 0:
            dets = np.asarray(dets)
            scores = dets[:, 4] if dets.shape[1] > 4 else np.ones(len(dets))
            best = dets[int(np.argmax(scores))]
            x1, y1, x2, y2 = map(int, best[:4])

            h, w = frame.shape[:2]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w - 1, x2), min(h - 1, y2)

            face_crop = frame[y1:y2, x1:x2]
            bbox = (x1, y1, x2, y2)

            if face_crop is not None and face_crop.size > 0:
                cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
                txt = f"Label: {current_label}"
                cv2.putText(vis, txt, (x1, max(0, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 3, cv2.LINE_AA)
                cv2.putText(vis, txt, (x1, max(0, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 1, cv2.LINE_AA)
        else:
            cv2.putText(vis, "No face detected", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

        cv2.putText(vis, help_text, (20, vis.shape[0]-20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        cv2.imshow("Collect (ESC exit)", vis)

        key = cv2.waitKey(1) & 0xFF
        if key == 27:
            break

        if key in label_map:
            current_label = label_map[key]

        do_save = False
        if key == ord('s'):
            do_save = True
        else:
            now = time.time()
            if now - last_save >= save_interval:
                do_save = True

        if do_save and face_crop is not None and bbox is not None and face_crop.size > 0:
            last_save = time.time()
            ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"{ts_str}_{current_label}_{uuid.uuid4().hex[:8]}.jpg"
            path = os.path.join(save_dir, filename)
            cv2.imwrite(path, face_crop)

            x1, y1, x2, y2 = bbox
            with open(label_csv, "a", newline="") as f:
                w = csv.writer(f)
                w.writerow([ts_str, filename, current_label, x1, y1, x2, y2])

            print(f"[Saved] {filename}  label={current_label}")

    cap.release()
    cv2.destroyAllWindows()


# =========================================================
# 4) 训练：DINOv3 embedding + SVM
# =========================================================
def train_svm(captures_dir: str, label_csv: str, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)

    paths, labels = [], []
    with open(label_csv, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            fn = row["filename"]
            emo = row["emotion_name"]
            p = os.path.join(captures_dir, fn)
            if os.path.exists(p):
                paths.append(p)
                labels.append(emo)

    if not paths:
        raise SystemExit("没读到训练数据：请检查 captures_dir / label_csv")

    embedder = DinoEmbedder()

    X = []
    keep_labels = []
    for p, lab in zip(paths, labels):
        bgr = cv2.imread(p)
        if bgr is None:
            continue
        X.append(embedder.embed_bgr(bgr))
        keep_labels.append(lab)

    X = np.asarray(X, dtype=np.float32)

    le = LabelEncoder()
    y_id = le.fit_transform(np.asarray(keep_labels))

    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=10.0, gamma="scale", probability=True)),
    ])
    clf.fit(X, y_id)

    joblib.dump(clf, os.path.join(out_dir, "dinov3_svm.joblib"))
    joblib.dump(le, os.path.join(out_dir, "label_encoder.joblib"))

    print("[OK] Saved SVM model to:", out_dir)
    print("Classes:", list(le.classes_))


# =========================================================
# 5) 推理：webcam 实时画框 + DINOv3+SVM 情绪标签
# =========================================================
def run_webcam(
    face_weights: str,
    svm_dir: str,
    cam_id: int = 0,
    conf_thres: float = 0.40,
):
    clf = joblib.load(os.path.join(svm_dir, "dinov3_svm.joblib"))
    le = joblib.load(os.path.join(svm_dir, "label_encoder.joblib"))

    det = FaceDetector(face_weights, device="cpu")
    embedder = DinoEmbedder()

    cap = cv2.VideoCapture(cam_id)
    if not cap.isOpened():
        raise SystemExit("Failed to open camera")

    cv2.namedWindow("DINOv3+SVM Emotion (ESC exit)", cv2.WINDOW_NORMAL)

    fps_t0 = time.time()
    frames = 0
    fps = 0.0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        vis = frame.copy()
        cropped_face, dets = det.get_face(frame)

        if dets is not None and len(dets) > 0:
            dets = np.asarray(dets)
            scores = dets[:, 4] if dets.shape[1] > 4 else np.ones(len(dets))
            best = dets[int(np.argmax(scores))]
            x1, y1, x2, y2 = map(int, best[:4])

            h, w = frame.shape[:2]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w - 1, x2), min(h - 1, y2)

            face_crop = frame[y1:y2, x1:x2]
            if face_crop.size > 0:
                emb = embedder.embed_bgr(face_crop).reshape(1, -1)
                prob = clf.predict_proba(emb)[0]
                idx = int(prob.argmax())
                score = float(prob[idx])
                label = le.inverse_transform([idx])[0]

                text = f"{label} ({score*100:.1f}%)"
                if score < conf_thres:
                    text = f"unknown ({score*100:.1f}%)"

                cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(vis, text, (x1, max(0, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3, cv2.LINE_AA)
                cv2.putText(vis, text, (x1, max(0, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
        else:
            cv2.putText(vis, "No face detected", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

        frames += 1
        if frames % 10 == 0:
            t1 = time.time()
            dt = t1 - fps_t0
            if dt > 0:
                fps = 10.0 / dt
            fps_t0 = t1

        cv2.putText(vis, f"FPS: {fps:.1f}", (20, vis.shape[0] - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        cv2.imshow("DINOv3+SVM Emotion (ESC exit)", vis)

        key = cv2.waitKey(1) & 0xFF
        if key == 27:
            break

    cap.release()
    cv2.destroyAllWindows()


# =========================================================
# 6) CLI
# =========================================================
def main():
    import argparse
    p = argparse.ArgumentParser()
    sub = p.add_subparsers(dest="cmd", required=False)

    p_collect = sub.add_parser("collect")
    p_collect.add_argument("--face-weights", default="./weights/Alignment_RetinaFace.pth")
    p_collect.add_argument("--save-dir", default="./data/captures")
    p_collect.add_argument("--label-csv", default="./data/labels.csv")
    p_collect.add_argument("--cam", type=int, default=0)
    p_collect.add_argument("--interval", type=float, default=2.0)

    p_train = sub.add_parser("train")
    p_train.add_argument("--captures", default="./data/captures")
    p_train.add_argument("--labels", default="./data/labels.csv")
    p_train.add_argument("--out", default="./svm_out")

    p_run = sub.add_parser("run")
    p_run.add_argument("--face-weights", default="./weights/Alignment_RetinaFace.pth")
    p_run.add_argument("--svm-dir", default="./svm_out")
    p_run.add_argument("--cam", type=int, default=0)
    p_run.add_argument("--conf", type=float, default=0.40)

    args, _unknown = p.parse_known_args()


    # ✅ 关键：没传 cmd 就默认 run
    if args.cmd is None:
        args.cmd = "run"
        # 给 run 的默认参数补齐
        args.face_weights = "./weights/Alignment_RetinaFace.pth"
        args.svm_dir = "./svm_out"
        args.cam = 0
        args.conf = 0.40

    if args.cmd == "collect":
        collect_webcam(
            face_weights=args.face_weights,
            save_dir=args.save_dir,
            label_csv=args.label_csv,
            cam_id=args.cam,
            save_interval=args.interval,
        )
    elif args.cmd == "train":
        train_svm(args.captures, args.labels, args.out)
    elif args.cmd == "run":
        run_webcam(
            face_weights=args.face_weights,
            svm_dir=args.svm_dir,
            cam_id=args.cam,
            conf_thres=args.conf,
        )


if __name__ == "__main__":
    main()


FileNotFoundError: [Errno 2] No such file or directory: './svm_out\\dinov3_svm.joblib'